# Hepatitis C Clinical Data Analysis
### Python exploratory data analysis for a GitHub portfolio

**Objective:** Explore demographic and laboratory patterns across blood donor, hepatitis, fibrosis and cirrhosis records.

> This project is for analytical demonstration only and is **not a medical diagnostic tool**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('data/HepatitisCdata.xlsx', sheet_name='HepatitisCdata')
df = df.rename(columns={'Column1':'Record_ID'})
df.head()

## 1. Data quality assessment

In [ ]:
print('Shape:', df.shape)
print('\nDuplicates:', df.duplicated().sum())
print('\nMissing values:\n', df.isna().sum())
print('\nCategory distribution:\n', df['Category'].value_counts())

### Data-quality findings
- 615 records.
- No duplicate rows.
- Missing laboratory values are concentrated mainly in ALP and CHOL.
- The dataset is highly imbalanced: most observations are Blood Donors.
- Keep missing clinical values as missing for descriptive analysis rather than replacing them with zero.

In [ ]:
df['Category_Code'] = df['Category'].str.extract(r'^([^=]+)')[0]
df['Category_Label'] = df['Category'].str.split('=', n=1).str[1]
df['Sex_Label'] = df['Sex'].map({'m':'Male','f':'Female'})
df['Disease_Status'] = np.where(
    df['Category_Label'].isin(['Hepatitis','Fibrosis','Cirrhosis']),
    'Disease','Non-disease'
)
df['Age_Group'] = pd.cut(df['Age'], [0,29,39,49,59,69,200], labels=['<30','30-39','40-49','50-59','60-69','70+'])

## 2. Clinical category distribution

In [ ]:
category_counts = df['Category_Label'].value_counts()
category_counts

In [ ]:
plt.figure(figsize=(10,6))
category_counts.sort_values().plot(kind='barh')
plt.title('Records by Clinical Category')
plt.xlabel('Number of Records')
plt.tight_layout()
plt.show()

## 3. Laboratory profiles by category

In [ ]:
labs = ['ALB','ALP','ALT','AST','BIL','CHE','CHOL','CREA','GGT','PROT']
lab_means = df.groupby('Category_Label')[labs].mean().round(2)
lab_means

In [ ]:
order = ['Blood Donor','suspect Blood Donor','Hepatitis','Fibrosis','Cirrhosis']
plt.figure(figsize=(10,6))
lab_means['AST'].reindex(order).plot(kind='bar')
plt.title('Average AST by Clinical Category')
plt.ylabel('Average AST')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
lab_means['GGT'].reindex(order).plot(kind='bar')
plt.title('Average GGT by Clinical Category')
plt.ylabel('Average GGT')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 4. Relationships between laboratory markers

In [ ]:
corr = df[labs + ['Age']].corr()
corr.round(3)

In [ ]:
plt.figure(figsize=(9,6))
for category in order:
    g = df[df['Category_Label'] == category]
    plt.scatter(g['AST'], g['GGT'], alpha=0.55, label=category)
plt.title('AST vs GGT by Clinical Category')
plt.xlabel('AST')
plt.ylabel('GGT')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Key findings
- The dataset contains **615 records**, with **533 Blood Donors**, 7 suspect donors, 24 Hepatitis, 21 Fibrosis and 30 Cirrhosis records.
- The classes are strongly imbalanced, so category percentages and raw counts should always be shown together.
- **AST, GGT and bilirubin (BIL)** show the largest standardized differences between disease and non-disease groups in this dataset.
- Cirrhosis records have substantially higher average **AST, GGT and creatinine**, while average **albumin and cholesterol** are lower than in Blood Donors.
- Male records have a higher disease-record proportion than female records in this sample; this is descriptive and should not be interpreted as population risk without a representative study design.
- Missing laboratory values should not be replaced with zero because zero can represent a real clinical measurement.
- These findings are associations in this dataset, not diagnostic thresholds or causal medical conclusions.